# Build the Performance Table
The Goal of this script is tack on performance metrics 

## Read in the data

In [ ]:
%cd /home/ian/Projects/pi-cropopt

import matplotlib.pyplot as plt
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
from pymoo.indicators.igd_plus import IGDPlus
from pymoo.indicators.igd import IGD
from pymoo.indicators.gd import GD
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from pandarallel import pandarallel
from pico.cropopt.VirtualFamers import  RangedVirtualFarmer

from pico.pinsga2.pinsga2 import PINSGA2


import itertools 
from multiprocessing import Pool


from matplotlib.animation import FuncAnimation
from IPython.display import HTML


run_file_names = [
                  "/rdata/ian/pico/paperRuns/finalRuns/global_run_table_1988.pkl", 
                  "/rdata/ian/pico/paperRuns/finalRuns/global_run_table_1989.pkl", 
                  "/rdata/ian/pico/paperRuns/finalRuns/global_run_table_1990.pkl",
                  "/rdata/ian/pico/paperRuns/finalRuns/global_run_table_1991.pkl",
                  "/rdata/ian/pico/paperRuns/finalRuns/global_run_table_1992.pkl",
                  "/rdata/ian/pico/paperRuns/finalRuns/global_run_table_1993.pkl",
                  "/rdata/ian/pico/paperRuns/finalRuns/global_run_table_1994.pkl",
                  "/rdata/ian/pico/paperRuns/finalRuns/global_run_table_1995.pkl",
                  "/rdata/ian/pico/paperRuns/finalRuns/global_run_table_1996.pkl",
                  "/rdata/ian/pico/paperRuns/finalRuns/global_run_table_1997.pkl",
                  "/rdata/ian/pico/paperRuns/finalRuns/global_run_table_1998.pkl",
                  "/rdata/ian/pico/paperRuns/finalRuns/global_run_table_1999.pkl",
                  #"/rdata/ian/pico/paperRuns/finalRuns/global_run_table_2000.pkl",
                  #"/rdata/ian/pico/paperRuns/finalRuns/global_run_table_2001.pkl",
                  #"/rdata/ian/pico/paperRuns/finalRuns/global_run_table_2002.pkl",
                  #"/rdata/ian/pico/paperRuns/finalRuns/global_run_table_2003.pkl",
                  "/rdata/ian/pico/paperRuns/finalRuns/global_run_table_2004.pkl",
                  "/rdata/ian/pico/paperRuns/finalRuns/global_run_table_2005.pkl",
                  "/rdata/ian/pico/paperRuns/finalRuns/global_run_table_2006.pkl", 
                  "/rdata/ian/pico/paperRuns/finalRuns/global_run_table_2007.pkl",
                  "/rdata/ian/pico/paperRuns/finalRuns/global_run_table_2008.pkl",
                  "/rdata/ian/pico/paperRuns/finalRuns/global_run_table_2009.pkl",
                  "/rdata/ian/pico/paperRuns/finalRuns/global_run_table_2010.pkl",
                  "/rdata/ian/pico/paperRuns/finalRuns/global_run_table_2011.pkl",
                  "/rdata/ian/pico/paperRuns/finalRuns/global_run_table_2012.pkl",
                  "/rdata/ian/pico/paperRuns/finalRuns/global_run_table_2013.pkl",
                  "/rdata/ian/pico/paperRuns/finalRuns/global_run_table_2014.pkl",
                  "/rdata/ian/pico/paperRuns/finalRuns/global_run_table_2015.pkl",
                  "/rdata/ian/pico/paperRuns/finalRuns/global_run_table_2016.pkl",
                  "/rdata/ian/pico/paperRuns/finalRuns/global_run_table_2016.pkl",
                  "/rdata/ian/pico/paperRuns/finalRuns/global_run_table_2017.pkl"
                  ]

#years = list(range(1988, 2017+1))
years = [ 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2004, 2005, 2006, 2007, 2008, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2016, 2017]

dm_range_lower = 30
dm_range_upper = 40
dm_range_str = f"{dm_range_lower}to{dm_range_upper}"

global_pf_file_names = [f"/rdata/ian/pico/paperRuns/finalRuns/global_pf_{year}.pkl" for year in years]

full_run_tabs = [pd.read_pickle(run_file_name) for run_file_name in run_file_names]

# Drop the variable columns
to_drop = ["var%d" % i for i in range(0, 67)]
[tab.drop(to_drop, axis=1, inplace=True) for tab in full_run_tabs]

full_run_tab = pd.concat(full_run_tabs, axis=0)

global_pfs = [pd.read_pickle(file) for file in global_pf_file_names]

global_pf = pd.concat(global_pfs, axis=0)

gd_inds = {
    year : GD(global_pf.loc[global_pf["year"] == year,("irr_total", "yield")].values)
    for year in years
    }

del global_pfs
del full_run_tabs



In [ ]:


# Set up DM agent
agent = RangedVirtualFarmer(dm_range_lower, dm_range_upper)


#pandarallel.initialize(progress_bar=False, nb_workers=24)

## Configuration Definition
What configurations do we want to measure against our baseline? 

In [ ]:
algorithm = ["nsga2",   "pinsga2"]
dm_range =  ["None",    dm_range_str ]
pop_size =  [120,       60       ]

configurations = [
    np.all([
        full_run_tab['algorithm'] == algorithm[c], 
        full_run_tab['DM_range']  == dm_range[c],
        full_run_tab['pop_size']  == pop_size[c]
        ], axis=0) 
    for c in range(len(algorithm))  

]


## Non-dominated filter 


In [ ]:
def nd_filter(run_record): 

    before = run_record.shape[0]

    # Non-dominated sort
    f1 = -run_record['yield'] # Made this negative because we to maximize yield
    f2 = run_record['irr_total']
    F = np.column_stack((f1,f2))

    opt_fronts = NonDominatedSorting().do(F, only_non_dominated_front=True)

    run_record = run_record.iloc[opt_fronts, :].copy()

    return run_record

## Performance metric functions
Different ways we can evaluate a given configuration 

### Final Generational Distance
Base peformance on the final generation of the optimization 


In [ ]:
def calc_gd(df): 

    years = set(df["year"])

    if len(years) != 1: 
        raise ValueError("Bad table given. Should only be one year. Got:" + str(years))

    year = years.pop()

    df_nd = nd_filter(df)

    df_nd['gd_final_gen'] = gd_inds[year](df_nd.loc[:,('yield','irr_total')].values)
    return df_nd.loc[:,('algorithm',  'gen', 'DM_range', 'pop_size', 'run', 'gd_final_gen', 'year')]


def finalGenGD(tab): 

    max_gen = max(tab['gen'])

    # Filter all but the final generations 
    tab = tab[tab["gen"] == max_gen]

    summary = tab.groupby(['algorithm', 'DM_range', 'pop_size', 'run', 'year']).apply(calc_gd)

    return summary.drop_duplicates(subset=['algorithm',  'gen', 'DM_range', 'pop_size', 'run', 'gd_final_gen', 'year'])




#### Prefered Final Solution 

In [ ]:


def calc_gd_preferred(df): 

    #df = nd_filter(df)

    years = set(df["year"])
    algorithms = set(df["algorithm"])
    dm_ranges = set(df["DM_range"])
    dm_range = dm_ranges.pop()

    if len(years) != 1: 
        raise ValueError("Bad table given. Should only be one year. Got:" + str(years))

    year = years.pop()
    if len(algorithms) != 1 or len(dm_ranges): 
        raise ValueError("Bad algorithm or dm_range found")
    algorithm = algorithms.pop()

    F = df.loc[:,('yield','irr_total')].values

    if algorithm == "pinsga2":

        [dm_range_lower, dm_range_upper] = dm_range.split("to")
        dm_range_lower = int(dm_range_lower)
        dm_range_upper = int(dm_range_upper)

        mask = np.logical_and(F[:, 1] >= dm_range_lower, F[:, 1] <= dm_range_upper)

        F_inbound = F[mask,:]

        if F_inbound.shape[0] != 0:

            max_yield = max(F_inbound[:,0])

        else: 

            max_yield = max(F[:,0])

        F = df.loc[df["yield"] == max_yield, ('yield', 'irr_total')].values 

        #df['gd_preferred'] = np.array2string(F[0])
        df['preferred_yield'] = F[0][0]
        df['preferred_irr_total'] = F[0][1]

    else: 

        df['preferred_yield'] = None
        df['preferred_irr_total'] = None


    return df.loc[:,('algorithm',  'gen', 'DM_range', 'pop_size', 'run', 'preferred_yield', 'preferred_irr_total', 'year')]


def preferredSolutionGD(tab):

    max_gen = max(tab['gen'])

    # Filter all but the final generations 
    tab = tab[tab["gen"] == max_gen]

    summary = tab.groupby(['algorithm', 'DM_range', 'pop_size', 'run', 'year']).apply(calc_gd_preferred)

    return summary.drop_duplicates(subset=['algorithm',  'gen', 'DM_range', 'pop_size', 'run', 'preferred_yield', 'preferred_irr_total', 'year'])


## Global GD 
Take all the solutions per run throughout the generations, and remove dominated solution

In [ ]:
def globalGD(tab): 

    run_identifiers = ['algorithm', 'DM_range', 'pop_size', 'run', 'year'] 

    tab = tab.groupby(run_identifiers).apply(nd_filter)
    # Filter all but the final generations 

    # TODO Shit's breaking here
    summary = tab.groupby(run_identifiers).apply(calc_gd)

    return summary.drop_duplicates(subset=['algorithm',  'gen', 'DM_range', 'pop_size', 'run', 'gd_final_gen', 'year'])


### Run the analysis 
The goal being that we compress each run into a given metric. So at the end we should have one row per run, with the given metrics calculated for that run

In [ ]:
def analyzeResults(args):
    year = args[0]
    config = args[1]
    print("Processing year and config %d-%d..." % (year, id(config)))
    #result = finalGenGD(full_run_tab[config])
    result = preferredSolutionGD(full_run_tab[config])
 
    print("Completed year and config %d-%d..." % (year, id(config)))

    return result

run_summary  = None 

run_summaries = []

# Approach 2 
args = list(itertools.product(* [years, configurations]))

len(list(args))
with Pool(20) as p: 
    run_summaries = p.map(analyzeResults, args)

run_summary = pd.concat(run_summaries)
run_summary


## Plot Pareto fronts

In [ ]:
years = set(run_summary['year'])

for year in years:

    fig, ax = plt.subplots(figsize=(8,6))
    ax.set_ylabel("Yield (kg/ha)")
    ax.set_xlabel("Total Irrigation (mm)")
    ax.set_title(f"PI-NSGA-II Vs NSGA-II in {year}")
    gpf = global_pf[global_pf["year"] == year]

    gpf = gpf.sort_values(by="yield")

    line = ax.plot(gpf["irr_total"], gpf["yield"],
                label="ND Solutions of 30 NSGA-II runs",
                color="red")



    select_mask = np.logical_and(run_summary["algorithm"] == "pinsga2" , run_summary["year"] == year)
    selected_points = run_summary.loc[select_mask, ("preferred_yield", "preferred_irr_total")].values

    ax.scatter(selected_points[:,1], selected_points[:,0], label="Final preferred solutions: 30 PI-NSGA-II runs")
    ax.legend() 



